# Opus MT English to Spanish Translation

**Model:** Helsinki-NLP/opus-mt-en-es | **Size:** 298MB | **Product:** prod-aqzhzecxfn42o

A MarianMT neural machine translation model from the Opus-MT project, trained on a large parallel corpus for high-quality English to Spanish translation. Part of the Helsinki-NLP collection, widely used in production translation pipelines.

## Use Cases
- Automated English to Spanish document translation
- Multilingual customer communication localization
- Content management systems with translation workflows
- E-commerce product description translation for Spanish-speaking markets

In [ ]:
import boto3
import sagemaker
from sagemaker import ModelPackage

region = boto3.Session().region_name
role = sagemaker.get_execution_role()
sm_client = boto3.client('sagemaker', region_name=region)

print(f'Region: {region}')
print(f'Role: {role}')

In [ ]:
# Replace with your actual Model Package ARN from AWS Marketplace
model_package_arn = 'arn:aws:sagemaker:REGION:ACCOUNT:model-package/MODEL_PACKAGE_NAME'

# Validate ARN before deploying
if 'REGION' in model_package_arn or 'ACCOUNT' in model_package_arn or 'MODEL_PACKAGE_NAME' in model_package_arn:
    raise ValueError(
        'model_package_arn contains placeholder values. '
        'Subscribe to the model on AWS Marketplace and replace with the actual ARN.'
    )

endpoint_name = 'opus-mt-en-es-translation'
instance_type = 'ml.m5.xlarge'

try:
    model = ModelPackage(
        role=role,
        model_package_arn=model_package_arn,
        sagemaker_session=sagemaker.Session()
    )
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type=instance_type,
        endpoint_name=endpoint_name
    )
    print(f'Endpoint deployed: {endpoint_name}')
except Exception as e:
    print(f'Deployment failed: {e}')
    raise

## Step 2: Run Inference

Send English text to translate to Spanish. The model handles sentences and short paragraphs effectively.

In [ ]:
import json

runtime = boto3.client('sagemaker-runtime', region_name=region)

# Sample English texts for translation
texts = [
    'Welcome to our online store. We offer free shipping on all orders over fifty dollars.',
    'Machine learning models are transforming how businesses automate complex tasks.',
    'Please contact our support team if you need assistance with your order.',
]

for text in texts:
    payload = json.dumps({'inputs': text})
    try:
        response = runtime.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType='application/json',
            Body=payload
        )
        result_raw = response['Body'].read().decode('utf-8')
        try:
            result = json.loads(result_raw)
            print(f'English: {text}')
            print(f'Spanish: {result}\n')
        except json.JSONDecodeError:
            print(f'Raw response: {result_raw}')
    except Exception as e:
        print(f'Inference failed: {e}')
        raise

In [ ]:
# Cleanup - delete the endpoint to avoid ongoing charges
try:
    sm_client.delete_endpoint(EndpointName=endpoint_name)
    print(f'Endpoint {endpoint_name} deleted.')
except Exception as e:
    print(f'Cleanup failed: {e}')